In [1]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU!")

from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/structscore_repro'
os.makedirs(OUT_DIR, exist_ok=True)

%cd /content
!git clone -q https://github.com/cs329yangzhong/discourse-driven-summary-factuality-evaluation
%cd /content/discourse-driven-summary-factuality-evaluation
!pip install -q "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main" fuzzywuzzy python-Levenshtein networkx

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

GPU: Tesla T4
Mounted at /content/drive
/content
/content/discourse-driven-summary-factuality-evaluation
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


True

In [2]:
import json, pandas as pd
from nltk.tokenize import sent_tokenize
from minicheck.minicheck import MiniCheck
from tqdm.auto import tqdm

df = pd.read_csv('data/diversumm_test.csv')
OUT_FILE = f'{OUT_DIR}/my_minicheck_scores.jsonl'

done = set()
if os.path.exists(OUT_FILE):
    with open(OUT_FILE) as f:
        done = {json.loads(line)['row'] for line in f}
print(f'Already done: {len(done)} / {len(df)}')

scorer = MiniCheck(model_name='flan-t5-large', cache_dir='./ckpts')

with open(OUT_FILE, 'a') as f:
    for i, r in tqdm(df.iterrows(), total=len(df)):
        if i in done:
            continue
        sents = sent_tokenize(r['summary'])
        # same setup as the paper: each summary sentence checked against the full document,
        # MiniCheck's default 500-word chunks, max over chunks
        _, probs, _, _ = scorer.score(docs=[r['doc']] * len(sents), claims=sents, chunk_size=500)
        f.write(json.dumps({'row': i, 'origin': r['origin'], 'scores': [float(p) for p in probs]}) + '\n')
        f.flush()
print('Finished!')

Already done: 1 / 451


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 3.13GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

... (8 outputs left out)

special_tokens_map.json:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

  0%|          | 0/451 [00:00<?, ?it/s]




Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]


Evaluating:  12%|█▎        | 1/8 [00:15<01:48, 15.49s/it]


Evaluating:  25%|██▌       | 2/8 [00:19<00:54,  9.00s/it]


Evaluating:  38%|███▊      | 3/8 [00:23<00:33,  6.73s/it]


Evaluating:  50%|█████     | 4/8 [00:28<00:23,  5.76s/it]


Evaluating:  62%|██████▎   | 5/8 [00:32<00:15,  5.15s/it]


Evaluating:  75%|███████▌  | 6/8 [00:36<00:09,  4.83s/it]


Evaluating:  88%|████████▊ | 7/8 [00:40<00:04,  4.50s/it]


Evaluating: 100%|██████████| 4/4 [00:02<00:00,  1.53it/s]

Finished!


In [3]:
import sys, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
sys.path.append('structscore')
import discourse_utils as du

df = pd.read_csv('data/diversumm_test.csv')
mine = pd.read_json(f'{OUT_DIR}/my_minicheck_scores.jsonl', lines=True).set_index('row')
df['my_scores'] = mine['scores']
assert df['my_scores'].notna().all(), 'Cell 2 has not finished all rows yet'

# 1) Sentence-level agreement with the authors' released scores
a, b = [], []
for _, r in df.iterrows():
    theirs = du.process_sent_score(str(r['segment_minicheck_score']))
    if len(theirs) == len(r['my_scores']):
        a += theirs; b += r['my_scores']
a, b = np.array(a), np.array(b)
print(f'Sentences compared: {len(a)}')
print(f'Pearson correlation with released scores: {np.corrcoef(a, b)[0,1]:.4f}')
print(f'Mean absolute difference: {np.abs(a-b).mean():.4f}\n')

# 2) AUC with your own scores, before and after re-weighting
splits = ['multinews', 'qmsum', 'GovReport', 'arXiv', 'ChemSum']
names  = ['MNW', 'QMS', 'GOV', 'AXV', 'CSM']
paper_orig = [57.67, 60.66, 83.24, 78.66, 59.74]   # Table 4, row 11
paper_rw   = [56.79, 60.36, 84.75, 79.38, 60.06]   # Table 4, row 12

rows = []
for i, s in enumerate(splits):
    d = df[df.origin == s]
    orig, rw, fails = [], [], 0
    for _, r in d.iterrows():
        sc = list(r['my_scores'])
        try:
            w, o, *_ = du.reweight_score(sc, r, alpha=1, depth_factor=1)
        except Exception:
            o = w = float(np.mean(sc)); fails += 1
        orig.append(o); rw.append(w)
    rows.append({'split': names[i],
                 'paper_orig': paper_orig[i], 'mine_orig': round(roc_auc_score(d.label, orig)*100, 2),
                 'paper_rw': paper_rw[i],     'mine_rw':   round(roc_auc_score(d.label, rw)*100, 2),
                 'align_failures': fails})

res = pd.DataFrame(rows)
print(res.to_string(index=False))
print('\nMacro:', res[['paper_orig','mine_orig','paper_rw','mine_rw']].mean().round(2).to_dict())
res.to_csv(f'{OUT_DIR}/levelB_results.csv', index=False)

Sentences compared: 3675
Pearson correlation with released scores: 1.0000
Mean absolute difference: 0.0000



/content/discourse-driven-summary-factuality-evaluation/structscore/discourse_utils.py:586: RuntimeWarning: invalid value encountered in sqrt
  depth = int(np.sqrt(int(end_id) - int(start_id)))


split  paper_orig  mine_orig  paper_rw  mine_rw  align_failures
  MNW       57.67      61.90     56.79    57.85               0
  QMS       60.66      60.66     60.36    58.12               0
  GOV       83.24      83.24     84.75    86.74               0
  AXV       78.66      78.66     79.38    79.52               0
  CSM       59.74      59.74     60.06    60.71               0

Macro: {'paper_orig': 67.99, 'mine_orig': 68.84, 'paper_rw': 68.27, 'mine_rw': 68.59}
